# 🌸 Laboratório de Segurança Federada no Google Colab (GPU T4 Ativa)

Este notebook executa o estudo empírico comparativo do **Artigo 1**: comparando o **Baseline Limpo**, o **Ataque Bruto** e os **Ataques Furtivos (Targeted Backdoor & Trigger Patch)** sob as 4 defesas convencionais (`FedAvg`, `FedMedian`, `Krum`, `Bulyan`).

---
### ⚡ Instruções Iniciais:
1. No menu superior: **Ambiente de execução (Runtime) -> Alterar tipo de ambiente de execução -> Selecionar T4 GPU -> Salvar**.
2. Execute as células abaixo em ordem.

## 1. Verificar Disponibilidade da GPU T4

In [ ]:
!nvidia-smi

## 2. Instalar Dependências do Flower e PyTorch

In [ ]:
!pip install -q "flwr[simulation]>=1.26.0" "flwr-datasets[vision]>=0.5.0" torch torchvision matplotlib numpy

## 3. Upload e Instalação Limpa do Projeto

In [ ]:
%cd /content
import os
import shutil
import zipfile
from google.colab import files

# Limpeza rigorosa de pastas e JSONs antigos
!rm -rf /content/quickstart-pytorch* /content/*.zip /content/*.json /content/resultados_ataque_furtivo*

print('Faça o upload do novo arquivo projeto_flower.zip (100% limpo):')
uploaded = files.upload()

for fn in uploaded.keys():
    with zipfile.ZipFile(fn, 'r') as zip_ref:
        zip_ref.extractall('/content')
    print(f'✔ {fn} extraído com sucesso!')

%cd /content/quickstart-pytorch
!pip install -q -e .
print('\n✔ PACOTE INSTALADO COM SUCESSO! Pronto para executar em GPU!')

## 4. Executar Bateria Comparativa em GPU T4

Executa os 4 cenários essenciais do Artigo 1:
1. **Baseline Limpo** (Sem Ataque - pr=0.0)
2. **Ataque Bruto** (Gaussian Noise em FedAvg e Bulyan)
3. **Ataque Furtivo: Targeted Backdoor** (FedAvg, FedMedian, Krum, Bulyan sob Non-IID alpha=0.1)
4. **Ataque Furtivo: Trigger Patch** (FedAvg e Bulyan)

In [ ]:
import subprocess
import time
import os

%cd /content/quickstart-pytorch

cenarios = [
    # 1. Baseline Limpo de Controle (Sem Ataque)
    {"defesa": "FedAvg", "ataque": "label_flipping", "pr": 0.0, "alpha": 0.1, "rounds": 10, "seed": 42},
    
    # 2. Ataques Brutos de Controle (Ruído Gaussiano)
    {"defesa": "FedAvg", "ataque": "gaussian_noise",  "pr": 0.4, "alpha": 0.1, "rounds": 10, "seed": 42},
    {"defesa": "Bulyan", "ataque": "gaussian_noise",  "pr": 0.4, "alpha": 0.1, "rounds": 10, "seed": 42},
    
    # 3. Ataques Furtivos: Targeted Backdoor (As 4 Defesas sob alpha=0.1)
    {"defesa": "FedAvg",    "ataque": "targeted_backdoor", "pr": 0.4, "alpha": 0.1, "rounds": 10, "seed": 42},
    {"defesa": "FedMedian", "ataque": "targeted_backdoor", "pr": 0.4, "alpha": 0.1, "rounds": 10, "seed": 42},
    {"defesa": "Krum",      "ataque": "targeted_backdoor", "pr": 0.4, "alpha": 0.1, "rounds": 10, "seed": 42},
    {"defesa": "Bulyan",    "ataque": "targeted_backdoor", "pr": 0.4, "alpha": 0.1, "rounds": 10, "seed": 42},
    
    # 4. Ataque Furtivo: Trigger Patch Físico
    {"defesa": "FedAvg", "ataque": "trigger_patch", "pr": 0.4, "alpha": 0.1, "rounds": 10, "seed": 42},
    {"defesa": "Bulyan", "ataque": "trigger_patch", "pr": 0.4, "alpha": 0.1, "rounds": 10, "seed": 42},
]

total = len(cenarios)
print(f"Iniciando Bateria com {total} experimentos em GPU T4...\n")

for i, exp in enumerate(cenarios, 1):
    d = exp['defesa']
    a = exp['ataque']
    pr = exp['pr']
    da = exp['alpha']
    r = exp['rounds']
    s = exp['seed']
    
    print(f"[{i:02d}/{total}] Iniciando: {d} | Ataque: {a} (PR={pr}, α={da}, Seed={s})...")
    run_cfg = f"defense_mode='{d}' attack_type='{a}' poison_rate={pr} dirichlet_alpha={da} num-server-rounds={r} seed={s}"
    cmd = f'flwr run . --stream --run-config "{run_cfg}"'
    
    res = subprocess.run(cmd, shell=True)
    subprocess.run("ray stop", shell=True, capture_output=True)
    time.sleep(2)

print("\n✔ Bateria experimental concluída com sucesso em GPU!")

## 5. Gerar Figuras Científicas, Matrizes de Confusão e Painel Resumo

In [ ]:
%cd /content/quickstart-pytorch
!python plotar_resultados.py

## 6. Baixar Todos os Resultados (.ZIP) para o seu Computador

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('/content/resultados_ataque_furtivo', 'zip', '/content/quickstart-pytorch/resultados_ataque_furtivo')
print('Baixando resultados_ataque_furtivo.zip para o seu computador...')
files.download('/content/resultados_ataque_furtivo.zip')